# 06 · Model comparison on the live replay

Reads the `replay.parquet` (and `test.parquet`) written by notebooks 03, 04 and 05 and compares
every model on the same rows: the (device, target, origin, lead) cells that every model
forecast. The replay is the live run of models frozen at 2026-09-01 00:00 Europe/Rome, one
168 h forecast every 6 h; a horizon H forecast is the lead 1..H prefix of that path. A model
is named `model (scope)`. Runs or models that are missing are skipped with a note. Every
setting lives in the first code cell, tagged `parameters` for papermill; figures and tables
are written to OUT_DIR.

In [ ]:
# Run folders relative to the repo root; each holds replay.parquet (+ test.parquet, summary.csv,
# best_params.json | best_order.json). A model found in two runs is taken from the first.
RUNS = [
    "runs/gx10/nb03/c13-per_device-h168",
    "runs/gx10/nb03/c13-pooled-h168",
    "runs/gx10/nb04/c13-sarima-h168",
    "runs/gx10/nb05/c13-chronos2-h168",
]
HORIZONS = [48, 96, 168]         # tables use leads 1..H of the same paths
REFERENCE = "naive168"           # skill is one minus the MAE ratio to this model
ORIGIN_HOUR_H = 48               # horizon of the origin-hour comparison
SMOOTH_H = 6                     # rolling window of the lead curves
AGE_BIN_H = 6                    # forecast-age bins of the revision plot: the replay stride
ALPHA = 0.05                     # Diebold-Mariano significance level
EXAMPLE_TARGET = "grid_import"   # revision example: target, device and window
EXAMPLE_DEVICE = None            # None: the device with the largest mean actual
EXAMPLE_DAYS = 3                 # days before the last origin, so every forecast age exists
LOCAL_TZ = "Europe/Rome"
OUT_DIR = "runs/nb06-c13"

In [ ]:
import json
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy import stats

T0 = time.time()
NB_DIR = Path.cwd().resolve()
if not (NB_DIR / "data").exists() and (NB_DIR / "notebooks" / "data").exists():
    NB_DIR = NB_DIR / "notebooks"
REPO_ROOT = next(p for p in [NB_DIR, *NB_DIR.parents] if (p / "pyproject.toml").exists())
OUT = REPO_ROOT / OUT_DIR
OUT.mkdir(parents=True, exist_ok=True)
pd.set_option("display.width", 220, "display.max_columns", 30, "display.precision", 4,
              "display.max_colwidth", None)

KEY = ["device_id", "target", "origin", "horizon"]
QS = [0.1, 0.25, 0.5, 0.75, 0.9]
# One colour per model in every figure; LightGBM keeps notebook 03's blue, naive lines dashed.
COLORS = {"lightgbm (per_device)": "#3b6fb6", "lightgbm (pooled)": "#2a9d8f",
          "sarima (per_device)": "#d9822b", "chronos2 (pooled)": "#c0392b",
          "chronos2_zeroshot (pooled)": "#8e44ad", "naive168 (per_device)": "#333333",
          "naive24 (per_device)": "#999999"}
SPARE = iter(["#e6ab02", "#66a61e", "#e7298a", "#a6761d", "#1b9e77", "#7570b3"])


def style(label):
    if label not in COLORS:
        COLORS[label] = next(SPARE)
    return {"color": COLORS[label], "ls": "--" if label.startswith("naive") else "-"}


def save(fig, name):
    fig.savefig(OUT / f"{name}.png", dpi=120, bbox_inches="tight")
    plt.show()


def to_csv(df, name):
    df.to_csv(OUT / f"{name}.csv")
    return df


print("repo:", REPO_ROOT, "| output:", OUT)

## 1 · Runs, coverage and the common rows

In [ ]:
replays, tests, choices = [], [], []
for run in RUNS:
    d = REPO_ROOT / run
    if not (d / "replay.parquet").exists():
        print(f"skip {run}: no replay.parquet")
        continue
    r = pd.read_parquet(d / "replay.parquet").assign(run=run)
    replays.append(r)
    if (d / "test.parquet").exists():
        t = pd.read_parquet(d / "test.parquet")
        if "model" not in t:  # notebook 03: one model per run, naive baselines as extra columns
            t = t.assign(model=r["model"].iloc[0], scope=r["scope"].iloc[0])
        tests.append(t.assign(run=run))
    for f in ("best_params.json", "best_order.json"):
        if (d / f).exists():
            chosen = json.loads((d / f).read_text())
            if not all(isinstance(v, dict) for v in chosen.values()):  # one set for all targets
                chosen = {"all": chosen}
            for target, p in chosen.items():
                choices.append({"run": run, "target": target,
                                "chosen": ", ".join(f"{k}={v}" for k, v in p.items())})
assert replays, "none of RUNS has a replay.parquet"


def combine(frames):
    """One frame with a `label` per model; a label found in two runs keeps the first run."""
    df = pd.concat(frames, ignore_index=True)
    df["label"] = df["model"] + " (" + df["scope"] + ")"
    first = df.drop_duplicates("label").set_index("label")["run"]
    other = df["run"] != df["label"].map(first)
    for lab, run in df.loc[other, ["label", "run"]].drop_duplicates().itertuples(index=False):
        print(f"note: {lab} also in {run}; kept {first[lab]}")
    return df[~other].drop_duplicates(KEY + ["label"]).dropna(subset=["actual"])


def common(df):
    """The rows every model forecast (non-NaN point), so all models are scored on the same cells."""
    df = df.dropna(subset=["point"])
    return df[df.groupby(KEY)["label"].transform("size") == df["label"].nunique()].copy()


ALL = combine(replays)
R = common(ALL)
R["ae"] = (R["point"] - R["actual"]).abs()
LABELS = sorted(R["label"].unique(), key=lambda s: (s.startswith("naive"), s))
TARGETS = sorted(R["target"].unique())
for lab in LABELS:
    style(lab)
REF = next((lab for lab in LABELS if lab.split(" (")[0] == REFERENCE), None)
if REF is None:
    print(f"note: reference {REFERENCE} not loaded, skill columns are empty")
HS = [h for h in HORIZONS if h <= R["horizon"].max()]
if HS != list(HORIZONS):
    print(f"note: horizons {sorted(set(HORIZONS) - set(HS))} skipped, common paths stop at "
          f"lead {R['horizon'].max()}")
HM = max(HS)

# Daylight = the site's is_daylight of that hour, from notebook 02.
DAY = pd.read_parquet(NB_DIR / "data" / "processed_hourly.parquet",
                      columns=["ts_hour", "is_daylight"]).groupby("ts_hour")["is_daylight"].max()
R["daylight"] = R["ts_hour"].map(DAY)
if R["daylight"].isna().any():
    print(f"note: {R['daylight'].isna().sum()} rows without is_daylight, "
          "left out of the daylight metrics")

local = lambda s: s.dt.tz_convert(LOCAL_TZ).dt.strftime("%Y-%m-%d %H:%M")
cov = ALL.groupby("label").agg(run=("run", "first"), rows=("actual", "size"),
                               devices=("device_id", "nunique"), origins=("origin", "nunique"),
                               first_origin=("origin", "min"), last_origin=("origin", "max"),
                               max_lead=("horizon", "max"))
cov[["first_origin", "last_origin"]] = cov[["first_origin", "last_origin"]].apply(local)
cov["scored"] = R.groupby("label").size().reindex(cov.index).fillna(0).astype(int)
cov["lost"] = cov["rows"] - cov["scored"]
display(to_csv(cov, "coverage"))
print(f"common rows per model: {len(R) // len(LABELS)} | devices {R['device_id'].nunique()} | "
      f"origins {R['origin'].nunique()} | targets {TARGETS}")
if choices:
    display(to_csv(pd.DataFrame(choices).set_index(["run", "target"]), "chosen_params"))

`lost` is what each model gives up to the common rows. The hyperparameters and orders are the
ones each notebook chose on its tuning folds.

## 2 · Headline table

In [ ]:
def metrics(df, by):
    """MAE, RMSE, mean pinball loss over QS and the 50% / 80% interval coverage per group."""
    y, e = df["actual"], df["point"] - df["actual"]
    pin = sum(np.maximum(q * (y - df[f"q{q}"]), (q - 1) * (y - df[f"q{q}"])) for q in QS) / len(QS)
    inside = lambda lo, hi: ((y >= df[lo]) & (y <= df[hi])).astype(float).where(df[lo].notna())
    x = df[by].assign(MAE=e.abs(), RMSE=e ** 2, pinball=pin,
                      cov50=inside("q0.25", "q0.75"), cov80=inside("q0.1", "q0.9"))
    out = x.groupby(by).mean()
    out["RMSE"] = np.sqrt(out["RMSE"])
    return out.assign(n=x.groupby(by).size())


def score(df):
    """Headline metrics per target and model: nMAE averages MAE / mean actual over devices."""
    by = ["target", "label"]
    out = metrics(df, by)
    dev = df.groupby(by + ["device_id"]).agg(ae=("ae", "mean"), y=("actual", "mean"))
    out["nMAE"] = (dev["ae"] / dev["y"]).replace(np.inf, np.nan).groupby(level=by).mean()
    ref = out["MAE"].xs(REF, level="label") if REF else pd.Series(dtype=float)
    out["skill"] = 1 - out["MAE"] / out.index.get_level_values("target").map(ref).to_numpy()
    return out


parts = []
for H in HS:
    sub = R[R["horizon"] <= H]
    parts.append(score(sub).assign(H=H, hours="all"))
    ex = sub[(sub["target"] == "grid_export") & (sub["daylight"] == 1)]
    if len(ex):
        parts.append(score(ex).assign(H=H, hours="daylight"))
HEAD = (pd.concat(parts).reset_index()
        .sort_values(["target", "hours", "H", "MAE"])
        .set_index(["target", "hours", "H", "label"])
        [["MAE", "RMSE", "nMAE", "skill", "pinball", "cov50", "cov80", "n"]])
display(to_csv(HEAD, "headline").round(4))

Sorted by MAE within each block (kWh/h). skill > 0 beats the naive 168 h forecast;
`daylight` repeats grid_export on the hours with the sun above the horizon, where all its
energy is. A calibrated model covers about 0.50 and 0.80.

## 3 · Accuracy vs lead time

In [ ]:
LEAD = R.groupby(["target", "label", "horizon"])["ae"].mean().unstack("label")
smooth = lambda s: s.rolling(SMOOTH_H, center=True, min_periods=1).mean()
fig, axes = plt.subplots(2, len(TARGETS), figsize=(7.5 * len(TARGETS), 8), squeeze=False,
                         sharex=True)
for j, target in enumerate(TARGETS):
    L = LEAD.loc[target]
    for lab in LABELS:
        axes[0, j].plot(L.index, L[lab], lw=0.7, alpha=0.25, **style(lab))
        axes[0, j].plot(L.index, smooth(L[lab]), lw=1.8, label=lab, **style(lab))
        if REF and lab != REF:
            axes[1, j].plot(L.index, 1 - L[lab] / L[REF], lw=0.7, alpha=0.25, **style(lab))
            axes[1, j].plot(L.index, 1 - smooth(L[lab]) / smooth(L[REF]), lw=1.8, **style(lab))
    axes[0, j].set(title=f"{target}: MAE by lead hour", ylabel="MAE (kWh/h)")
    axes[1, j].set(title=f"{target}: skill vs {REFERENCE} by lead hour", xlabel="lead (h)",
                   ylabel="1 - MAE / MAE ref")
    axes[1, j].axhline(0, color="k", lw=0.8)
    for ax in axes[:, j]:
        ax.set_xticks(range(0, int(L.index.max()) + 1, 24))
        ax.grid(alpha=0.3)
axes[0, 0].legend(fontsize=8)
fig.tight_layout()
save(fig, "lead_curve")
_ = to_csv(LEAD, "lead_mae")

Faint: MAE of each lead hour; bold: its 6 h rolling mean. The daily cycle is the hour of day
the lead lands on; the trend is how fast each model loses its edge over the naive forecast.

## 4 · Per device

In [ ]:
DEVM = metrics(R[R["horizon"] <= HM], ["target", "device_id", "label"])["MAE"].unstack("label")
rows = []
n_dev = DEVM.index.get_level_values("device_id").nunique()
fig, axes = plt.subplots(1, len(TARGETS), figsize=(7 * len(TARGETS), 0.45 * n_dev + 2.5),
                         squeeze=False)
for ax, target in zip(axes[0], TARGETS):
    M = DEVM.loc[target][LABELS]
    if REF is None:
        ax.set_visible(False)
        continue
    S = (1 - M.div(M[REF], axis=0)).drop(columns=REF)
    rows.append(pd.concat({"MAE": M, "skill": S}, axis=1).assign(target=target))
    v = min(1.0, np.nanmax(np.abs(S.to_numpy())))
    im = ax.imshow(S.to_numpy(), cmap="RdBu", vmin=-v, vmax=v, aspect="auto")
    for (i, k), val in np.ndenumerate(S.to_numpy()):
        ax.text(k, i, f"{val:+.2f}", ha="center", va="center", fontsize=8,
                color="white" if abs(val) > 0.6 * v else "black")
    ax.set_xticks(range(S.shape[1]), S.columns, rotation=30, ha="right")
    ax.set_yticks(range(S.shape[0]), S.index)
    ax.set_title(f"{target}: skill vs {REFERENCE}, H={HM}")
    fig.colorbar(im, ax=ax, shrink=0.8)
fig.tight_layout()
save(fig, "device_skill")
if rows:
    DEVS = pd.concat(rows).reset_index().set_index(["target", "device_id"])
    display(to_csv(DEVS, "device_scores").round(3))

Blue beats the naive forecast on that device, red loses to it. A model that wins on average
but is red on a few devices is a candidate for a per-device fallback.

## 5 · Origin hour

In [ ]:
HO = ORIGIN_HOUR_H if ORIGIN_HOUR_H in HS else min(HS)
sub = R[R["horizon"] <= HO]
sub = sub.assign(origin_hour=sub["origin"].dt.tz_convert(LOCAL_TZ).dt.hour)
OH = metrics(sub, ["target", "origin_hour", "label"])["MAE"].unstack("label")[LABELS]
fig, axes = plt.subplots(1, len(TARGETS), figsize=(7 * len(TARGETS), 4), squeeze=False)
w = 0.8 / len(LABELS)
for ax, target in zip(axes[0], TARGETS):
    T = OH.loc[target]
    for i, lab in enumerate(LABELS):
        ax.bar(np.arange(len(T)) + (i - (len(LABELS) - 1) / 2) * w, T[lab], w,
               color=style(lab)["color"], label=lab)
    ax.set_xticks(range(len(T)), [f"{h:02d}:00" for h in T.index])
    ax.set(title=f"{target}: MAE by origin hour (local), H={HO}", ylabel="MAE (kWh/h)")
    ax.grid(alpha=0.3, axis="y")
axes[0, 0].legend(fontsize=8)
fig.tight_layout()
save(fig, "origin_hour")
display(to_csv(OH, "origin_hour").round(4))

At H=48 an origin's leads land on different hours of day, so this shows which issue time
starts closest to the hard hours (for export, the morning ramp).

## 6 · Forecast revision: how the age of a forecast changes accuracy

In [ ]:
# Each target hour is forecast by every origin in the 168 h before it, one per 6 h age bin.
# Keep the target hours that every bin forecast, so each point averages the same hours.
AGE = R.assign(age=((R["horizon"] - 1) // AGE_BIN_H + 1) * AGE_BIN_H)
nbins = AGE.groupby(["device_id", "target", "ts_hour"])["age"].transform("nunique")
AGE = AGE[nbins == AGE["age"].nunique()]
REV = AGE.groupby(["target", "label", "age"])["ae"].mean().unstack("label")
fig, axes = plt.subplots(1, len(TARGETS), figsize=(7.5 * len(TARGETS), 4), squeeze=False)
for ax, target in zip(axes[0], TARGETS):
    for lab in LABELS:
        ax.plot(REV.loc[target].index - AGE_BIN_H / 2, REV.loc[target][lab], marker="o", ms=3,
                label=lab, **style(lab))
    ax.invert_xaxis()
    ax.set(title=f"{target}: MAE vs forecast age ({AGE['ts_hour'].nunique()} target hours)",
           xlabel="forecast age at the target hour (h, 6 h bins) → issued later",
           ylabel="MAE (kWh/h)")
    ax.grid(alpha=0.3)
axes[0, 0].legend(fontsize=8)
fig.tight_layout()
save(fig, "revision_age")
to_csv(REV, "revision_age")

# Example: successive forecasts of the same hours by the best model, older ones fainter.
ex_t = EXAMPLE_TARGET if EXAMPLE_TARGET in TARGETS else TARGETS[0]
best = HEAD.loc[(ex_t, "all", HM)]["MAE"].idxmin()
ex = R[(R["label"] == best) & (R["target"] == ex_t)]
devs = ex.groupby("device_id")["actual"].mean().sort_values(ascending=False).index
dev = EXAMPLE_DEVICE if EXAMPLE_DEVICE in devs else devs[0]
ex = ex[ex["device_id"] == dev]
end = ex["origin"].max().tz_convert(LOCAL_TZ).floor("D")
win = ex[(ex["ts_hour"] >= end - pd.Timedelta(days=EXAMPLE_DAYS)) & (ex["ts_hour"] < end)]
orig = sorted(win["origin"].unique())
fig, ax = plt.subplots(figsize=(15, 4))
for i, (o, g) in enumerate(win.groupby("origin")):
    ax.plot(g["ts_hour"].dt.tz_convert(LOCAL_TZ), g["point"], color=COLORS[best], lw=1,
            alpha=0.08 + 0.8 * i / max(len(orig) - 1, 1))
act = win.drop_duplicates("ts_hour").sort_values("ts_hour")
ax.plot(act["ts_hour"].dt.tz_convert(LOCAL_TZ), act["actual"], color="k", lw=2, label="actual")
ax.plot([], [], color=COLORS[best], label=f"{best}: {len(orig)} forecasts, older = fainter")
ax.set(title=f"{ex_t}, {dev}: successive forecasts of the same hours", ylabel="kWh/h")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
save(fig, "revision_example")

Read the first plot left to right as the delivery hour approaching: a flat line means new
data does not help (a model driven by calendar and weather), a line that drops in the last
bins means the recent lags carry information. The example shows the spread of the successive
forecasts around the actual.

## 7 · Significance: Diebold–Mariano

In [ ]:
def dm_pvalue(d, h):
    """Two-sided DM test of mean(d) = 0: Bartlett HAC variance with lag h-1, Harvey correction."""
    d = d.dropna().to_numpy()
    T = len(d)
    lag = min(h - 1, T - 1)
    dc = d - d.mean()
    gam = [dc[k:] @ dc[:T - k] / T for k in range(lag + 1)]
    var = gam[0] + 2 * sum((1 - k / (lag + 1)) * gam[k] for k in range(1, lag + 1))
    corr = (T + 1 - 2 * h + h * (h - 1) / T) / T
    if T < 3 or var <= 0 or corr <= 0:
        return np.nan
    return 2 * stats.t.sf(abs(d.mean() / np.sqrt(var / T) * np.sqrt(corr)), df=T - 1)


# One loss per origin (mean absolute error over devices and leads <= H); successive origins
# overlap, so the HAC lag counts origin steps: h = ceil(H / stride) steps ahead.
STRIDE = pd.Series(np.sort(R["origin"].unique())).diff().median() / pd.Timedelta(hours=1)
DM = []
for target in TARGETS:
    for H in HS:
        sub = R[(R["target"] == target) & (R["horizon"] <= H)]
        loss = sub.groupby(["origin", "label"])["ae"].mean().unstack()
        h = int(np.ceil(H / STRIDE))
        for a in LABELS:
            for b in LABELS:
                if a != b:
                    DM.append({"target": target, "H": H, "model": a, "vs": b,
                               "mae_diff": loss[a].mean() - loss[b].mean(),
                               "p": dm_pvalue(loss[a] - loss[b], h), "origins": len(loss)})
COLS = ["target", "H", "model", "vs", "mae_diff", "p", "origins"]
DM = to_csv(pd.DataFrame(DM, columns=COLS).set_index(COLS[:4]), "dm_tests")

# Matrix: row model vs column model; green = row significantly better, red = significantly worse.
VERDICT = plt.matplotlib.colors.ListedColormap(["#e8a1a1", "#f2f2f2", "#9fd49f"])
if len(LABELS) < 2:
    print("one model only: no pairs to test")
else:
    fig, axes = plt.subplots(len(TARGETS), len(HS), squeeze=False,
                             figsize=(4.8 * len(HS), 4.2 * len(TARGETS)))
    for (i, target), (j, H) in [(t, h) for t in enumerate(TARGETS) for h in enumerate(HS)]:
        D = DM.loc[(target, H)]
        P, S = (D[c].unstack().reindex(index=LABELS, columns=LABELS)
                for c in ("p", "mae_diff"))
        ax = axes[i, j]
        ax.imshow(np.where(P < ALPHA, np.sign(-S), 0), cmap=VERDICT, vmin=-1, vmax=1)
        for (r, c), p in np.ndenumerate(P.to_numpy()):
            ax.text(c, r, "" if r == c else f"{p:.3f}", ha="center", va="center", fontsize=8)
        ax.set_xticks(range(len(LABELS)), LABELS, rotation=35, ha="right", fontsize=8)
        ax.set_yticks(range(len(LABELS)), LABELS, fontsize=8)
        ax.set_title(f"{target}, H={H}: DM p-values")
    fig.tight_layout()
    save(fig, "dm_pvalues")

for (target, H), D in DM.groupby(level=["target", "H"]):
    win = D[(D["p"] < ALPHA) & (D["mae_diff"] < 0)].reset_index()
    print(f"{target} H={H} ({D['origins'].iloc[0]} origins): " + ("; ".join(
        f"{r.model} < {r.vs} (p={r.p:.3f})" for r in win.itertuples())
        or f"no difference at {ALPHA:.0%}"))

Each p-value tests equal mean absolute error of the row and column model on the per-origin
loss series. `A < B` means A is significantly more accurate at 5% (no multiple-test
correction, so read borderline values with care).

## 8 · Test (August) vs replay (September)

In [ ]:
if not tests:
    print("no test.parquet in RUNS: skipped")
else:
    TT = common(combine(tests))
    TT = TT[TT["horizon"] <= HM].assign(ae=lambda d: (d["point"] - d["actual"]).abs())
    tm = metrics(TT, ["target", "label"])["MAE"]
    rm = HEAD.xs(("all", HM), level=["hours", "H"])["MAE"]
    TV = pd.concat({"test_MAE": tm, "replay_MAE": rm}, axis=1).dropna()
    TV = TV.reindex([(t, lab) for t in TARGETS for lab in LABELS]).dropna()
    if REF:
        for blk in ("test", "replay"):
            ref = TV[f"{blk}_MAE"].xs(REF, level="label")
            base = TV.index.get_level_values("target").map(ref).to_numpy()
            TV[f"{blk}_skill"] = 1 - TV[f"{blk}_MAE"] / base
    display(to_csv(TV, "test_vs_replay").round(4))
    fig, axes = plt.subplots(1, len(TARGETS), figsize=(7 * len(TARGETS), 4), squeeze=False)
    for ax, target in zip(axes[0], TARGETS):
        T = TV.loc[target] if target in TV.index.get_level_values(0) else TV.iloc[:0]
        x = np.arange(len(T))
        colors = [style(lab)["color"] for lab in T.index]
        ax.bar(x - 0.2, T["test_MAE"], 0.4, color=colors, alpha=0.45, hatch="//",
               label="test (Aug)")
        ax.bar(x + 0.2, T["replay_MAE"], 0.4, color=colors, label="replay (Sep)")
        ax.set_xticks(x, T.index, rotation=30, ha="right")
        ax.set(title=f"{target}: MAE test vs replay, H={HM}", ylabel="MAE (kWh/h)")
        ax.grid(alpha=0.3, axis="y")
    axes[0, 0].legend(fontsize=8)
    fig.tight_layout()
    save(fig, "test_vs_replay")

Hatched: the August test folds (daily origins, each run on its own common rows); solid: the
September replay. The seasons differ, so compare the skill columns: a ranking that holds from
test to replay means the tuning period chose well.

## 9 · Summary

In [ ]:
lines = []
for (target, H), g in HEAD.xs("all", level="hours").groupby(level=["target", "H"]):
    g = g.droplevel(["target", "H"]).sort_values("MAE")
    best, line = g.index[0], ""
    if len(g) > 1:
        second = g.index[1]
        p = DM.loc[(target, H, best, second), "p"]
        line = f" | vs {second}: p={p:.3f}, {'significant' if p < ALPHA else 'not significant'}"
    lines.append(f"{target:12s} H={H:<3d} best {best:26s} MAE {g['MAE'].iloc[0]:.4f}  "
                 f"skill vs {REFERENCE} {g['skill'].iloc[0]:+.1%}{line}")
text = "\n".join(lines)
(OUT / "summary.txt").write_text(text + "\n")
print(text)
print(f"\n{len(LABELS)} models, {len(R) // len(LABELS)} common rows each, "
      f"outputs in {OUT} ({time.time() - T0:.0f} s)")